In [ ]:
# EDA

import duckdb
import pandas as pd
import time

# 파일 경로 지정
parquet_file = r"../data/ST4000DM000_v3.parquet"

# 판다스 출력 제한 해제 (모든 컬럼과 행을 숨김없이 표시)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

print(f"[{parquet_file}]")
print("종합 EDA 및 데이터 무결성 검증을 시작합니다 ...\n")
start_time = time.time()

# DuckDB 인메모리 연결
con = duckdb.connect()

try:
    # ---------------------------------------------------------
    # 1. 데이터 규격 (행/열 개수)
    # ---------------------------------------------------------
    print("=== [1. 데이터 규격 확인] ===")
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{parquet_file}')").fetchone()[0]
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    col_names = schema_df['column_name'].tolist()
    
    print(f"총 행 수(Rows): {total_rows:,} 개")
    print(f"총 열 수(Columns): {len(col_names)} 개\n")

    # ---------------------------------------------------------
    # 2. 상위 5개 데이터 샘플 (모든 열 표시)
    # ---------------------------------------------------------
    print("=== [2. 상위 5개 데이터 샘플 (생략 없음)] ===")
    sample_df = con.execute(f"SELECT * FROM read_parquet('{parquet_file}') LIMIT 5").fetchdf()
    print(sample_df)
    print("\n")

    # ---------------------------------------------------------
    # 3. 하드디스크 개체 및 클래스 분포 통계 (접미사 제거 로직 반영)
    # ---------------------------------------------------------
    print("=== [3. 하드디스크 개체 및 클래스 분포 통계] ===")
    
    # REGEXP_REPLACE(serial_number, '_\\d+$', '') : 시리얼 끝의 '_1', '_2' 등을 제거하여 동일 개체로 인식
    status_query = f"""
        SELECT 
            COUNT(DISTINCT REGEXP_REPLACE(serial_number, '_\\d+$', '')) AS total_objects,
            COUNT(DISTINCT CASE WHEN failure = 1 THEN REGEXP_REPLACE(serial_number, '_\\d+$', '') ELSE NULL END) AS failed_objects,
            COUNT(*) AS total_rows,
            SUM(CAST(failure AS INTEGER)) AS target_1_rows,
            SUM(CASE WHEN failure = 0 THEN 1 ELSE 0 END) AS target_0_rows
        FROM read_parquet('{parquet_file}')
    """
    stats = con.execute(status_query).fetchdf().iloc[0]

    total_obj = stats['total_objects']
    failed_obj = stats['failed_objects']
    healthy_obj = total_obj - failed_obj
    
    print("[개체 단위 통계 (물리적인 하드디스크 개수)]")
    print(f"- 전체 고유 개체 수 (접미사 통합): {int(total_obj):,} 개")
    print(f"- 정상 작동 하드: {int(healthy_obj):,} 개 ({(healthy_obj/total_obj)*100:.2f}%)")
    print(f"- 고장 발생 개체: {int(failed_obj):,} 개 ({(failed_obj/total_obj)*100:.2f}%)")
    
    # 0으로 나누기 방지
    if failed_obj > 0:
        print(f"- 개체 단위 비율 (Class 1 : 0) = 1 : {healthy_obj / failed_obj:.2f}\n")
    else:
        print("- 고장 발생 개체가 없어 비율을 계산할 수 없습니다.\n")

    print("[행 단위 클래스 분포 (❗진짜 ML 모델이 학습할 Target 레이블 비율)]")
    total_r = stats['total_rows']
    target_1 = stats['target_1_rows']
    target_0 = stats['target_0_rows']
    
    print(f"- 총 데이터 행 수: {int(total_r):,} 개")
    print(f"- Class 0 (정상인 날): {int(target_0):,} 개 ({target_0/total_r*100:.2f}%)")
    print(f"- Class 1 (고장 임박): {int(target_1):,} 개 ({target_1/total_r*100:.2f}%)")
    
    if target_1 > 0:
        ratio = target_0 / target_1
        print(f"- 실제 타겟 데이터 불균형 비율 (Class 1 : 0) = 1 : {ratio:.1f}\n")

    # ---------------------------------------------------------
    # 4. 모든 열에 대한 결측치(NULL) 개수 세기 (전체 열 표시)
    # ---------------------------------------------------------
    print("=== [4. 컬럼별 결측치 집계 (전체 열)] ===")
    null_count_selects = [f"COUNT(*) - COUNT(\"{col}\") AS \"{col}\"" for col in col_names]
    query_nulls = f"SELECT {', '.join(null_count_selects)} FROM read_parquet('{parquet_file}')"
    null_counts = con.execute(query_nulls).fetchdf().iloc[0]
    
    null_summary = pd.DataFrame({'Missing_Count': null_counts})
    null_summary['Missing_Ratio(%)'] = (null_summary['Missing_Count'] / total_rows) * 100
    
    # 필터링 없이 정렬만 수행하여 모든 열을 보여줌
    null_summary_sorted = null_summary.sort_values(by='Missing_Count', ascending=False)
    
    pd.set_option('display.max_rows', None)
    print(null_summary_sorted)
    pd.reset_option('display.max_rows')
    print("\n")

    # ---------------------------------------------------------
    # 5. 열 별 간단한 기초 통계 (최솟값, 최댓값, 평균, 표준편차)
    # ---------------------------------------------------------
    print("=== [5. 열 별 간단한 기초 통계 (Numeric Data)] ===")
    summary_df = con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    stats_df = summary_df[['column_name', 'column_type', 'min', 'max', 'avg', 'std']].copy()
    
    pd.set_option('display.max_rows', None)
    print(stats_df)
    pd.reset_option('display.max_rows')
    print("\n")

    # ---------------------------------------------------------
    # 6. 시계열 연속성 검사 (Date Gap)
    # ---------------------------------------------------------
    print("=== [6. 시계열 연속성(Date Gap) 검사] ===")
    gap_check_query = f"""
        WITH DateRange AS (
            SELECT 
                serial_number,
                MIN(CAST(date AS DATE)) as start_date,
                MAX(CAST(date AS DATE)) as end_date,
                COUNT(*) as actual_row_count,
                (MAX(CAST(date AS DATE)) - MIN(CAST(date AS DATE)) + 1) as expected_row_count
            FROM read_parquet('{parquet_file}')
            GROUP BY serial_number
        )
        SELECT 
            COUNT(*) AS serials_with_gaps,
            SUM(expected_row_count - actual_row_count) AS total_missing_days
        FROM DateRange
        WHERE actual_row_count != expected_row_count
    """
    gap_result = con.execute(gap_check_query).fetchdf().iloc[0]
    
    if gap_result['serials_with_gaps'] == 0:
        print("✅ 모든 개체의 날짜가 하루도 빠짐없이 연속적입니다.")
    else:
        print(f"⚠️ 날짜 공백(Gap)이 발견된 개체 수: {int(gap_result['serials_with_gaps']):,} 개")
        print(f"⚠️ 총 누락된 날짜(데이터 행) 수: {int(gap_result['total_missing_days']):,} 일")

except Exception as e:
    print(f"❌ 검증 중 오류 발생: {e}")
finally:
    con.close()
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    
    end_time = time.time()
    print(f"\n모든 종합 검증 완료. 총 소요 시간: {end_time - start_time:.2f}초")


[C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet]
종합 EDA 및 데이터 무결성 검증을 시작합니다 ...

=== [1. 데이터 규격 확인] ===
총 행 수(Rows): 79,698,388 개
총 열 수(Columns): 27 개

=== [2. 상위 5개 데이터 샘플 (생략 없음)] ===
  serial_number       date  smart_3_raw  smart_4_raw  smart_5_raw  smart_9_raw  smart_10_raw  smart_183_raw  smart_184_raw  smart_187_raw  smart_189_raw  smart_191_raw  smart_192_raw  smart_193_raw  smart_197_raw  smart_198_raw  smart_199_raw  smart_241_raw  smart_242_raw  Total_Reads  seek_error_count  Total_Seeks  failure  timeout_total  timeout_5s  smart_190_raw  smart_194_raw
0    S3008532_1 2014-04-19            0            2            0          562             0              1              0              0              1              0              1           1541              0              0              0     2721830352     8918128525    101001152                 0     29220189        0              0           0             19             19
1    S3008532_1 2014-04-20   

In [4]:
import duckdb
import pandas as pd

# 데이터 경로 설정
file_path = r'C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet'

# DuckDB 연결 (메모리 모드)
con = duckdb.connect()

print(f"데이터 분석 시작: {file_path}")

# 1. 원본 시리얼(Base)과 파생 번호(Suffix) 분리 및 기초 통계 쿼리
query = f"""
WITH base_info AS (
    SELECT 
        serial_number as full_serial,
        CASE 
            WHEN regexp_matches(serial_number, '.*_[0-9]+$') 
            THEN regexp_extract(serial_number, '^(.*)_[0-9]+$', 1)
            ELSE serial_number 
        END as base_serial,
        CASE 
            WHEN regexp_matches(serial_number, '.*_[0-9]+$') 
            THEN CAST(regexp_extract(serial_number, '_([0-9]+)$', 1) AS INTEGER)
            ELSE 0 
        END as suffix,
        count(*) as row_count
    FROM read_parquet('{file_path}')
    GROUP BY full_serial
)
SELECT 
    count(DISTINCT base_serial) as total_physical_entities,
    count(DISTINCT full_serial) as total_segments,
    max(suffix) as max_suffix_value,
    round(avg(row_count), 2) as avg_segment_lifespan,
    min(row_count) as min_segment_lifespan,  -- 가장 짧은 수명 추가
    round(count(DISTINCT full_serial) * 1.0 / count(DISTINCT base_serial), 2) as avg_segments_per_entity
FROM base_info
"""

# 결과 실행 및 출력
result = con.execute(query).df()

print("\n=== [개체 확인 결과] ===")
print(f"1. 전체 물리적 개체 수 (Base): {result['total_physical_entities'][0]:,} 개")
print(f"2. 분리된 총 세그먼트 수: {result['total_segments'][0]:,} 개")
print(f"3. 가장 높은 접미사 가중치 (_n): {result['max_suffix_value'][0]}")
print(f"4. 분리된 세그먼트들의 평균 수명: {result['avg_segment_lifespan'][0]:,.2f} 일")
print(f"5. 가장 짧은 세그먼트 수명: {result['min_segment_lifespan'][0]:,} 일") # 출력 추가
print(f"6. 개체당 평균 분리 횟수: {result['avg_segments_per_entity'][0]} 회")

# 추가: 가장 많이 쪼개진 상위 5개 개체 확인
print("\n=== [가장 많이 분리된 상위 5개 개체] ===")
top_split_query = f"""
WITH base_info AS (
    SELECT 
        serial_number,
        CASE 
            WHEN regexp_matches(serial_number, '.*_[0-9]+$') 
            THEN regexp_extract(serial_number, '^(.*)_[0-9]+$', 1)
            ELSE serial_number 
        END as base_serial
    FROM read_parquet('{file_path}')
    GROUP BY serial_number
)
SELECT 
    base_serial, 
    count(*) as split_count
FROM base_info
GROUP BY base_serial
ORDER BY split_count DESC
LIMIT 5
"""
top_splits = con.execute(top_split_query).df()
print(top_splits)


데이터 분석 시작: C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet

=== [개체 확인 결과] ===
1. 전체 물리적 개체 수 (Base): 37,024 개
2. 분리된 총 세그먼트 수: 88,491 개
3. 가장 높은 접미사 가중치 (_n): 14
4. 분리된 세그먼트들의 평균 수명: 900.64 일
5. 가장 짧은 세그먼트 수명: 1 일
6. 개체당 평균 분리 횟수: 2.39 회

=== [가장 많이 분리된 상위 5개 개체] ===
  base_serial  split_count
0    W3009EJ4            9
1    W300BGZ4            9
2    W300CTY8            9
3    W300SP1V            8
4    W300BJG2            8


In [ ]:
#  결측된 연속 날짜(행) 길이의 분포

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

check_p = "C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet"

print("🔍 시계열 연속성: 결측 구간 축약(Binning) 분석 및 시각화")

con = duckdb.connect()

try:
    gap_sql = f"""
        WITH date_diffs AS (
            SELECT 
                serial_number,
                date,
                LAG(date) OVER (PARTITION BY serial_number ORDER BY date) as prev_date
            FROM read_parquet('{check_p}')
        ),
        gaps AS (
            SELECT 
                serial_number,
                date_diff('day', CAST(prev_date AS DATE), CAST(date AS DATE)) - 1 AS missing_days
            FROM date_diffs
            WHERE prev_date IS NOT NULL
              AND date_diff('day', CAST(prev_date AS DATE), CAST(date AS DATE)) > 1
        )
        SELECT 
            missing_days AS missing_len,
            COUNT(*) AS gap_count
        FROM gaps
        GROUP BY missing_days
        ORDER BY missing_days
    """
    
    raw_df = con.execute(gap_sql).fetchdf()
    
    if len(raw_df) == 0:
        print("✅ 완벽한 연속성: 모든 개체에 대해 시계열 누락이 전혀 존재하지 않습니다.")
    else:
        # ✔️ 구간(Bin) 설정하여 데이터 축약
        bins = [0, 1, 2, 3, 7, 30, 90, float('inf')]
        labels = ['1일', '2일', '3일', '4~7일', '8~30일', '31~90일', '91일 이상']
        
        # cut 함수를 이용해 설정한 구간별로 라벨링
        raw_df['구간'] = pd.cut(raw_df['missing_len'], bins=bins, labels=labels, right=True)
        
        # 구간별 합계 집계
        agg_df = raw_df.groupby('구간', observed=False)['gap_count'].sum().reset_index()
        agg_df.columns = ['결측 기간(구간)', '발생 횟수']
        
        # 1. 요약 및 표 출력
        total_gaps = raw_df['gap_count'].sum()
        max_missing_len = raw_df['missing_len'].max()
        
        print("\n=== 📊 요약 ===")
        print(f"총 구간 단절 발생 횟수: {total_gaps:,}회")
        print(f"최장 결측일: 연속 {max_missing_len:,}일 누락")
        print("======================\n")
        
        print("▼ 깔끔하게 축약된 결측 기간별 발생 빈도 표")
        display(agg_df.style.hide(axis="index"))
        
        # 2. 축약된 그래프 시각화 (딱 7개의 막대로 직관적 표현)
        sns.set_theme(style="whitegrid")
        plt.figure(figsize=(10, 5))
        
        ax = sns.barplot(
            data=agg_df, 
            x='결측 기간(구간)', 
            y='발생 횟수', 
            palette='Blues_r'  # 색상 톤 조절
        )
        
        plt.title('Distribution of Missing Sequences (Grouped by Period)', fontsize=15, fontweight='bold', pad=15)
        plt.xlabel('Length of Missing Days (Binned)', fontsize=11)
        plt.ylabel('Frequency (Count)', fontsize=11)
        
        # 각 막대 위에 예쁘게 콤마(,) 포함된 수치 표시
        for p in ax.patches:
            height = p.get_height()
            if height > 0:
                ax.text(
                    p.get_x() + p.get_width() / 2., 
                    height + (max(agg_df['발생 횟수']) * 0.02), 
                    f'{int(height):,}', 
                    ha="center", va="bottom", fontsize=10, color='black', fontweight='bold'
                )
                
        # 라벨이 그래프를 뚫고 나가지 않게 y축 상단 여백 추가
        plt.ylim(0, max(agg_df['발생 횟수']) * 1.15)
        
        plt.tight_layout()
        plt.show()

except Exception as e:
    print(f"❌ 데이터 검증 중 오류: {e}")
finally:
    con.close()


In [6]:
# 변수 분포 확인 

import duckdb
import pandas as pd
import time

# 파일 경로 지정
parquet_file = r"C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet"

# 판다스 출력 제한 해제
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_rows', None)

print(f"[{parquet_file}]")
print("=== [열 별 요약 통계 (최소, 사분위수, 최대, 표준편차)] ===\n")
start_time = time.time()

# DuckDB 인메모리 연결
con = duckdb.connect()

try:
    # SUMMARIZE를 통해 전체 요약 통계 추출
    summary_df = con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{parquet_file}')").fetchdf()
    
    # 필요한 컬럼만 추출 (q25: 1사분위, q50: 중간값, q75: 3사분위)
    stats_df = summary_df[['column_name', 'column_type', 'min', 'q25', 'q50', 'q75', 'max', 'std']].copy()
    
    # 데이터프레임 출력
    print(stats_df)

except Exception as e:
    print(f"❌ 검증 중 오류 발생: {e}")
finally:
    con.close()
    
    # 설정 초기화
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.max_rows')
    
    end_time = time.time()
    print(f"\n통계 추출 완료. 총 소요 시간: {end_time - start_time:.2f}초")

[C:/Workspace/06_ML_projdect/26_1_COIN/data/ST4000DM000_v3.parquet]
=== [열 별 요약 통계 (최소, 사분위수, 최대, 표준편차)] ===



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         column_name column_type         min          q25           q50           q75              max                   std
0      serial_number     VARCHAR    S3000A9T         None          None          None         Z307Y2X9                  None
1               date        DATE  2013-05-10   2016-10-09    2018-06-21    2021-02-15       2025-03-13                  None
2        smart_3_raw      BIGINT           0            0             0             0            10590    12.868347441463923
3        smart_4_raw      BIGINT           1            5             9            13            30092    135.17048733606134
4        smart_5_raw      BIGINT           0            0             0             0            65488      312.220958773963
5        smart_9_raw      BIGINT           0        13793         28022         46544            78175     20269.12116244326
6       smart_10_raw      BIGINT           0            0             0             0           262144    32.128707161750896
